# Fill-rate exploration — Distributor B

This notebook reads Distributor B's raw fill-rate workbook and describes it, before any cleaning. It changes nothing and writes nothing. Text values (state, sales org, vendor, brand, product name) are counted, never printed.

**How to read it**

1. Start with **Read this first**: the main answers, each with the section that backs it.
2. Then open the part you need. Every numbered section has the same shape: the question, how to read the output, the code, and **What it shows** right under the output.
3. To run it, use the `.venv` kernel, with the repository root as working folder (in VS Code: setting `jupyter.notebookFileRoot` = `${workspaceFolder}`). The figures quoted in the text were produced by this code on 2026-10-22.

**Contents**

- **Read this first**
- **Part 1 — The file as received:** 1 Load · 2 Shape, grain, blanks, period, units · 3 Ordered, confirmed, shipped · 4 Where the missed cases sit · 5 Monthly chart · 6 Checks
- **Part 2 — The client's deck and deeper cuts:** 7 What the client's deck says · 8 Column identities · 9 Row average or case average · 10 Rows that did not ship in full but score 100% · 11 One sales org or several · 12 The weakest months · 13 Persistence · 14 New product ids · 15 Concentration · 16 Unit · 17 Keys and scope · 18 Checks
- **Part 3 — Before any cleaning:** proposals for the team, open questions
- **Part 4 — Where the lost cases are (2026-10-26):** concentration, repeat, months. Reads the clean working file.

## Read this first

### The file in four answers

| Question | Answer | See |
|---|---|---|
| What is one row? | One SAP product id, in one sales org, in one month, except 14 keys that appear twice (28 rows, 241 cases), each time across a product-name change. | 2, 17 |
| Which period? | August 2024 to August 2026: 25 months, none missing. | 2 |
| Which unit? | Cases, often fractional. Physical cases, confirmed by the client on 2026-10-23. The 9L factor is a separate column, not a replacement. | 2, 16 |
| What not to trust yet? | The 14 duplicated keys, and the product name as a key: 11 ids carry more than one name. | 2, 17 |

### How well Distributor B serves

- **The client's measure** (deck, slide 10): fill rate = 1 − cases unshipped / cases ordered, summed over cases. National target: 97%.
- **On this file:** 94.3% over the whole period and **93.0% for January–August 2026**. The deck gives no figure for Distributor B to compare with. → section 7
- **Confirmation fill: 94.4%.** 21,110 of 373,828 cases ordered (5.6%) were never confirmed. Shipped follows confirmed within a few cases. → section 3
- **Same definition as Distributor A.** Column N follows the deck's formula on 7,527 of 7,562 rows. → sections 7, 8
- **Do not average column N over rows.** That gives 90.3%. → section 9

### Where the missed cases are

- **On a few ids.** 10 ids carry 57% of missed cases; 8 ids make half. → sections 4, 15
- **In four bad months.** March 2025, July 2025, January 2026 and February 2026 are under 90% and hold 36% of missed cases. All four are small months by volume. March 2025 comes from the chronic ids, the other three from a few large ids. → sections 4, 12
- **On new products.** Ids first seen after August 2024 carry 19% of volume but 44% of missed cases, and confirm only 79.1% in their first three months. → section 14
- **Often in both sales orgs at once (73%).** Both orgs are in the same state, so this means little until we know whether they share a warehouse. → section 11
- **Mostly partial confirms.** 82% of missed cases are on rows partly confirmed, 18% on rows confirmed at zero. → section 3

### What the team must decide before cleaning

Three choices change the numbers: the fill-rate definition, the unit, and the key (with the 14 duplicated keys). Proposals in Part 3; questions in `docs/open_questions.md`.

### Words used in this notebook

| Word | Meaning here |
|---|---|
| Missed cases | Cases ordered − cases confirmed, never below zero. Called `short` in the code. |
| Confirmation fill | Σ cases confirmed / Σ cases ordered. The share of volume the distributor accepted. |
| Shipment fill | Σ cases shipped / Σ cases ordered. On this file Cases Shipped is never blank and stays within a few cases of Cases Confirmed. |
| Deck fill rate | 1 − Σ cases unshipped / Σ cases ordered: the client's definition, slide 10 of its introduction deck. |
| Column N | The file's Case Fill Rate column: the deck fill rate for one row. Averaging it over rows is misleading (section 9). |
| Id, org | SAP product id (column G), sales org (column C). An id-month is one id in one month, all orgs together. |
| Chronic id | An id with missed cases in six months or more. |

**Update 2026-10-23, after the client calls.** The figures in this notebook are unchanged. Physical cases and the case-weighted fill rate are now confirmed. Column G is the within-file id; on this distributor it matches the distributor item list in Master Data, not the client SAP SKU. The 14 duplicate keys are summed in the clean working file, not here. Both sales orgs map to one market in the location list; whether they share a warehouse is still open. Decisions: `docs/decisions.md`.

**Analysis, 2026-10-26.** Part 4 reads the clean working file. It says where the lost cases sit, whether the same ids miss again, and which months carry the loss. The figures in Parts 1 to 3 are unchanged.

**Template rebuild, 2026-11-04.** The current output is `data/cleaned/distributor_b_v1.xlsx`. Part 5 reads it. Parts 1 to 4 stay the record of the October work. The grain, the OOS measure and the duplicate-key rule are the 2026-11-04 decision in `docs/decisions.md`.


## Part 1 — The file as received

Written on 2026-10-22 from the raw file alone, before reading any client document.

### 1. Load

**What happens.** The workbook is chosen by folder, not by file name (file names contain real names). Columns are renamed by position, because the header text differs between distributors. Nothing in the file is changed.

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Letter of the distributor folder: data/raw/distributor_a or distributor_b.
DISTRIBUTOR = "b"

folder = Path("data/raw") / f"distributor_{DISTRIBUTOR}"
files = sorted(
    p for p in folder.iterdir()
    if p.suffix.lower() == ".xlsx" and not p.name.startswith("~$")
)
if len(files) != 1:
    raise RuntimeError(f"Expected one workbook in {folder}, found {len(files)}")

raw = pd.read_excel(files[0], sheet_name=0)
if raw.shape[1] != 14:
    raise RuntimeError(f"Expected 14 columns, found {raw.shape[1]}")

# Rename by position. Header text is not identical across distributors.
df = raw.copy()
df.columns = [
    "month", "state", "org", "vendor", "brand", "product", "sku",
    "ordered", "confirmed", "shipped", "unshipped", "pick_omits", "oos", "fill",
]
df["short"] = (df["ordered"] - df["confirmed"]).clip(lower=0)

def cases(value):
    return f"{value:,.0f}"

def pct(value):
    return f"{value:.1%}"

print(f"Distributor {DISTRIBUTOR.upper()}: {len(df):,} rows, {df.shape[1] - 1} source columns")
print("Raw file was only read. No cleaning is applied in this notebook.")


### 2. Shape, grain, blanks, period, units

**Question.** What is one row? How complete is each column? Which months? Which unit?

**How to read the output.** The first lines test the key (month × org × id) and count text values without printing them. The blank / zero table keeps the two apart on purpose: a blank is not a zero.

In [ ]:
keys = ["month", "org", "sku"]
print("Extra rows for a month x sales org x product id that already exists:", int(df.duplicated(keys).sum()))
print("States:", df["state"].nunique(), "| sales orgs:", df["org"].nunique())
orgs_per_state = df.groupby("state")["org"].nunique()
states_per_org = df.groupby("org")["state"].nunique()
print("States with more than one sales org:", int((orgs_per_state > 1).sum()))
print("Sales orgs spanning more than one state:", int((states_per_org > 1).sum()))
print("Vendors:", df["vendor"].nunique(), "| brands:", df["brand"].nunique())
print("Product names:", df["product"].nunique(), "| product ids:", df["sku"].nunique())
names_per_id = df.groupby("sku")["product"].nunique()
ids_per_name = df.groupby("product")["sku"].nunique()
print("Ids with more than one product name:", int((names_per_id > 1).sum()),
      "| max names per id:", int(names_per_id.max()))
print("Names with more than one id:", int((ids_per_name > 1).sum()),
      "| max ids per name:", int(ids_per_name.max()))

print("\nBlank cells (not the same as zero)")
for col in ["ordered", "confirmed", "shipped", "unshipped", "pick_omits", "oos", "fill"]:
    print(f"  {col:12} blank {int(df[col].isna().sum()):6}   zero {int((df[col] == 0).sum()):6}")

period = pd.PeriodIndex.from_fields(year=df["month"] // 100, month=df["month"] % 100, freq="M")
full = pd.period_range(period.min(), period.max(), freq="M")
missing = [str(p) for p in full if p not in set(period)]
print(f"\nPeriod {period.min()} to {period.max()} | months {period.nunique()} | gaps {missing or 'none'}")

ordered = df["ordered"].dropna()
share_sixths = float(np.mean(np.abs(ordered * 6 - np.round(ordered * 6)) < 0.02))
print(f"Ordered quantities that are a multiple of 1/6: {share_sixths:.1%}")
print("No unit conversion is applied.")


**What it shows**

- 14 month × org × id keys appear twice (section 17). Otherwise one row is one id in one org in one month.
- One state, two sales orgs. 11 ids carry more than one product name.
- No blank anywhere: every quantity column is filled, with zeros where nothing was missed.
- 25 months, no gap. 87% of ordered quantities are multiples of 1/6 of a case.

### 3. Ordered, confirmed, shipped

**Question.** How many cases were ordered, confirmed and shipped, and what does column N actually measure?

**How to read the output.** Totals first, then two ways of reading service (confirmation fill and column N), then the cases that fall between the columns.

In [ ]:
ordered = df["ordered"].sum()
confirmed = df["confirmed"].sum()
short = df["short"].sum()
print("Cases ordered            ", cases(ordered))
print("Cases confirmed          ", cases(confirmed))
print("Not confirmed            ", cases(short), pct(short / ordered))
print("Confirmation fill        ", pct(1 - short / ordered))
print("Unweighted mean of column N", pct(df["fill"].mean()))
print("Rows scored at 100%      ", pct((df["fill"] >= 0.999).mean()))

shipped_known = df["shipped"].notna()
ship_gap = (df["ordered"] - df["shipped"]).clip(lower=0)
print("Shipment gap where shipped is filled in", cases(ship_gap[shipped_known].sum()))
unknown = df["shipped"].isna() & (df["confirmed"] > 0)
zero_and_blank = df["shipped"].isna() & (df["confirmed"] == 0)
print("Ordered cases with no shipped qty and nothing confirmed", cases(df.loc[zero_and_blank, "ordered"].sum()))
print("Ordered cases confirmed but with no shipped qty", cases(df.loc[unknown, "ordered"].sum()))

blank_unshipped = df["unshipped"].isna()
if blank_unshipped.any() and (df.loc[blank_unshipped, "fill"] == 1).all():
    scored_full = blank_unshipped & ((df["ordered"] - df["shipped"]) > 0.05) & df["shipped"].notna()
    hidden = float((df.loc[scored_full, "ordered"] - df.loc[scored_full, "shipped"]).sum())
    print("Fill rate is 1 on every row where Cases Unshipped is blank.")
    present = ~blank_unshipped
    pred = 1 - df.loc[present, "unshipped"] / df.loc[present, "ordered"]
    matched = int(np.isclose(df.loc[present, "fill"], pred, atol=1e-6).sum())
    print(f"Where unshipped is filled in, fill rate equals 1 - unshipped/ordered on {matched:,} of {int(present.sum()):,} rows.")
    print("Cases not shipped on rows the fill-rate column scores at 100%:", cases(hidden))
else:
    gap = (df["fill"] - df["shipped"] / df["ordered"]).abs()
    print("Rows where fill rate is within 0.5 point of shipped/ordered:",
          f"{int((gap <= 0.005).sum()):,} of {len(df):,}")

print("Pick omit cases", cases(df["pick_omits"].sum(skipna=True)))
print("Cases OOS, blank cells skipped", cases(df["oos"].sum(skipna=True)))
zero = df["confirmed"] == 0
partial = (df["confirmed"] > 0) & (df["confirmed"] + 0.01 < df["ordered"])
print("Missed cases on zero-confirmed lines", cases(df.loc[zero, "short"].sum()),
      pct(df.loc[zero, "short"].sum() / short))
print("Missed cases on partial lines", cases(df.loc[partial, "short"].sum()),
      pct(df.loc[partial, "short"].sum() / short))


**What it shows**

- 373,828 cases ordered; 21,110 (5.6%) never confirmed: **confirmation fill 94.4%**.
- Shipped follows confirmed: the shipment gap is 21,185 cases. Column N matches shipped / ordered within half a point on 7,548 of 7,562 rows.
- Pick omits are tiny (76 cases). Cases OOS (21,401) is about the confirmation gap.
- 82% of missed cases are partial confirms, 18% are rows confirmed at zero.

### 4. Where the missed cases sit

**Question.** Which months, sales orgs, ids and brands carry the missed cases?

**How to read the output.** The monthly table gives cases ordered, missed cases (`short`), rows, and confirmation fill in % (`wfill`). Lists of shares are sorted from largest to smallest. Names are never printed, only ranks.

In [ ]:
monthly = (
    df.groupby("month")
    .agg(ordered=("ordered", "sum"), short=("short", "sum"), rows=("ordered", "size"))
)
monthly["wfill"] = 1 - monthly["short"] / monthly["ordered"]
view = monthly.copy()
view.index = view.index.astype(int)
print((view.assign(wfill=(view["wfill"] * 100).round(1))
       .round({"ordered": 0, "short": 0})).to_string())
print("Weakest confirmation fill", pct(monthly["wfill"].min()),
      "in", int(monthly["wfill"].idxmin()))
low = monthly[monthly["wfill"] < 0.90]
print("Months under 90%:", "none" if low.empty else int(low.shape[0]))
for month, row in low.sort_values("wfill").iterrows():
    print(f"  {int(month)}  fill {pct(row.wfill)}  ordered {cases(row.ordered)}  "
          f"({row.ordered / df['ordered'].sum():.1%} of volume)  missed {cases(row.short)}")

months = list(monthly.index)
cutoff = months[len(months) // 2]
early = df[df["month"] < cutoff]
late = df[df["month"] >= cutoff]
print("Before", int(cutoff), pct(1 - early["short"].sum() / early["ordered"].sum()))
print("From", int(cutoff), pct(1 - late["short"].sum() / late["ordered"].sum()))

def ranked_shares(column):
    totals = df.groupby(column)["short"].sum().sort_values(ascending=False)
    return [round(float(v / totals.sum()), 3) for v in totals]

print("Shortfall share by sales org, largest first:", ranked_shares("org"))
print("Shortfall share by state, largest first:", ranked_shares("state"))

org = df.groupby("org").agg(ordered=("ordered", "sum"), short=("short", "sum"))
org = org.sort_values("ordered", ascending=False)
org["fill"] = 1 - org["short"] / org["ordered"]
print("Confirmation fill by sales org, largest volume first:",
      [round(float(v), 3) for v in org["fill"]])
print("Order share by sales org, largest volume first:",
      [round(float(v), 3) for v in org["ordered"] / org["ordered"].sum()])

by_sku = df.groupby("sku")["short"].sum().sort_values(ascending=False)
print("Product ids with any missed cases:", int((by_sku > 0.05).sum()), "of", int(by_sku.size))
print("Share of missed cases in the 10 largest ids:", pct(by_sku.head(10).sum() / by_sku.sum()))
print("Share of missed cases in the 20 largest ids:", pct(by_sku.head(20).sum() / by_sku.sum()))

months_short = df[df["short"] > 0.05].groupby("sku")["month"].nunique()
chronic = months_short[months_short >= 6].index
once = months_short[months_short == 1].index
print("Ids missing in 6 months or more:", len(chronic),
      "| share of missed cases", pct(df.loc[df["sku"].isin(chronic), "short"].sum() / by_sku.sum()))
print("Ids missing in exactly one month:", len(once),
      "| share of missed cases", pct(df.loc[df["sku"].isin(once), "short"].sum() / by_sku.sum()))

by_brand = df.groupby("brand")["short"].sum().sort_values(ascending=False)
print("Brands with any missed cases:", int((by_brand > 0.05).sum()), "of", df["brand"].nunique())
print("Share of missed cases in the 5 largest brands:", pct(by_brand.head(5).sum() / by_brand.sum()))


**What it shows**

- **Four months are under 90%:** January 2026 (76.6%, but only 1.4% of volume), March 2025 (81.0%), February 2026 (83.5%, the largest miss in cases: 2,590) and July 2025 (86.5%).
- The two orgs are close: 52% and 48% of orders, 54% and 46% of missed cases.
- 322 of 391 ids miss at least once. The 10 largest carry 57%. Ids that miss in six months or more carry 41%: less chronic than Distributor A.
- Five brands carry 85% of missed cases.

### 5. Monthly chart

**What it draws.** Confirmation fill by month. The grey line is the whole-period value. The axis starts at 70%, so a weak month stays visible.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(range(len(monthly)), monthly["wfill"].to_numpy(), marker="o", ms=3.5)
overall = 1 - df["short"].sum() / df["ordered"].sum()
ax.axhline(overall, color="0.45", lw=1)
ax.set_xticks(range(len(monthly)))
ax.set_xticklabels(monthly.index.astype(int).astype(str), rotation=90)
ax.set_ylim(0.70, 1.01)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _pos: f"{value:.0%}"))
ax.set_ylabel("Confirmed cases / ordered cases")
ax.set_title(f"Distributor {DISTRIBUTOR.upper()} — confirmation fill by month")
fig.tight_layout()
plt.show()


**What it shows**

- Fill swings much more than on Distributor A, from 76.6% to 99.4%.
- Ordered volume swings too, from 5,391 to 23,167 cases a month. The dips fall in small months (section 12).

### 6. Checks

**What happens.** A few sanity counts, then asserts that pin the file this notebook was written against. If the client sends a refreshed extract, this cell fails on purpose, and the figures in the text must be checked again.

In [ ]:
id_len = df["sku"].astype("string").str.len()
print("Product-id length counts:", {int(k): int(v) for k, v in id_len.value_counts().sort_index().items()})
short_id = id_len < 7
print("Rows with an id shorter than 7 digits:", int(short_id.sum()),
      "| ordered cases", cases(df.loc[short_id, "ordered"].sum()))
dup_rows = df[df.duplicated(["month", "org", "sku"], keep=False)]
print("Rows on a duplicated month x org x id key:", len(dup_rows),
      "| ordered cases", cases(dup_rows["ordered"].sum()))
print("Rows with confirmed above ordered:", int((df["confirmed"] > df["ordered"] + 0.01).sum()))
print("Rows with shipped above ordered:", int((df["shipped"] > df["ordered"] + 0.01).sum()))
print("Rows with a negative quantity:", int((df[["ordered", "confirmed", "shipped", "unshipped", "pick_omits", "oos"]].min(axis=1) < 0).sum()))
print("Largest ordered quantity on one row:", cases(df["ordered"].max()))

# Pin the file this notebook was written against. A refreshed extract should fail here.
if DISTRIBUTOR == "a":
    assert len(df) == 29118
    assert int(df["month"].min()) == 202409 and int(df["month"].max()) == 202608
    assert abs(df["short"].sum() - 67812.6) < 1
elif DISTRIBUTOR == "b":
    assert len(df) == 7562
    assert int(df["month"].min()) == 202408 and int(df["month"].max()) == 202608
    assert abs(df["short"].sum() - 21110) < 1
else:
    raise AssertionError(DISTRIBUTOR)
print("Checks passed. File still matches the 2026-10-22 exploration.")


**What it shows**

- All ids have 7 digits. 28 rows sit on duplicated keys (241 cases).
- No row confirms or ships more than was ordered; no quantity is negative. The largest row is 5,863 cases.
- The asserts pass: the file is the one this notebook was written against.

## Part 2 — The client's deck and deeper cuts

Added on 2026-10-22, after reading the client's introduction deck, the only client document found. Same raw file, nothing cleaned. Section 7 checks the deck against the file. Sections 8 to 17 dig into the questions Part 1 left open.

### 7. What the client's deck says

**Question.** What does the client's own document say about these files, and does the file agree?

**The source.** `Introduction Presentation 2026.09.25.pdf`, in the `Client docs` folder of the team Drive. It is the only client document found on 2026-10-22: no meeting minutes, and `Working files` is empty.

- **Slide 2.** In 2025 the client began receiving monthly out-of-stock reports from its distributors. Objective: use them to find the SKUs and markets where DDMRP buffers can be adjusted, and set up a review process for the replenishment models.
- **Slide 7.** The OOS review receives the prior month's data on the 5th of each month. The client's root-cause analysis usually finds under-forecasting or late inventory arrival.
- **Slide 8.** Sample rows with the same 14 columns as these files. Challenges listed: each distributor has its own SKUs and slightly different columns; some offer vintage substitutions; an item can show as OOS when it is really discontinued, a back vintage or a limited-time offer.
- **Slide 10.** "Fill Rate Calculation 1-(Cases Unshipped/Cases Ordered)". National fill-rate target 97%; Distributor A "currently" at 95.7%. A table for one product by state, and an unlabeled row of monthly figures.

**What the deck does not give.** The unit of the cases (physical or 9-liter), the meaning of a blank cell, a definition of Pick Omits or Cases OOS, the aggregation key, the sales-org scope, any link to Master Data or Planning.

**How to read the output.** The first lines apply the deck's formula to three periods. On Distributor A, a second block finds the product shown on slide 10 from its numbers alone (names are never typed) and compares the slide's monthly row with three columns. Everything here stays a working hypothesis until the client confirms it.

In [ ]:
# Slide 10 of the client's introduction deck (2026-09-25): "Fill Rate Calculation 1-(Cases Unshipped/Cases Ordered)".
# Summed over cases. With this formula a blank Cases Unshipped adds nothing, as a zero would.
unshipped0 = df["unshipped"].fillna(0)

def deck_fill(mask):
    return 1 - unshipped0[mask].sum() / df.loc[mask, "ordered"].sum()

all_rows = df["month"] > 0
ytd = df["month"] >= 202601
last12 = df["month"] >= months[-12]
print("Deck formula, whole file         ", pct(deck_fill(all_rows)))
print("Deck formula, last 12 months     ", pct(deck_fill(last12)))
print("Deck formula, Jan-Aug 2026       ", pct(deck_fill(ytd)))
print("Confirmation fill, Jan-Aug 2026  ", pct(1 - df.loc[ytd, "short"].sum() / df.loc[ytd, "ordered"].sum()))
same_as_deck = (df["fill"] - (1 - unshipped0 / df["ordered"])).abs() <= 0.005
print(f"Rows where column N equals the deck formula within 0.5 point: {int(same_as_deck.sum()):,} of {len(df):,}")

if DISTRIBUTOR == "a":
    # Slide 10 table for one product: cases ordered and cases unshipped per state, sorted by cases ordered.
    # Names are left out on purpose. The product id is found from the numbers alone.
    deck_rows = np.array([[76, 9], [234, 75], [263, 44], [446, 28], [595, 115], [751, 184]])
    # Second table on the same slide: one unlabeled figure per month, Jan to Aug 2026.
    deck_months = np.array([165, 6, 21, 22, 20, 32, 76, 70])
    in_ytd = df[ytd].assign(unshipped0=unshipped0[ytd])
    best_sku, best_gap = None, np.inf
    for sku, g in in_ytd.groupby("sku"):
        by_state = g.groupby("state")[["ordered", "unshipped0"]].sum()
        if len(by_state) != len(deck_rows):
            continue
        gap = np.abs(by_state.sort_values("ordered").to_numpy() - deck_rows).sum()
        if gap < best_gap:
            best_sku, best_gap = sku, gap
    g = in_ytd[in_ytd["sku"] == best_sku]
    by_state = g.groupby("state")[["ordered", "unshipped0"]].sum().sort_values("ordered").to_numpy()
    # Rounded half up, as the deck's spreadsheet would display them.
    exact = int((np.floor(by_state + 0.5) == deck_rows).all(axis=1).sum())
    print(f"\nClosest product id, Jan-Aug 2026: {exact} of 6 state rows equal the deck to the case")
    print("  ordered", cases(by_state[:, 0].sum()), "vs deck", cases(deck_rows[:, 0].sum()),
          "| unshipped", cases(by_state[:, 1].sum()), "vs deck", cases(deck_rows[:, 1].sum()),
          "| sales orgs", g["org"].nunique())
    for label, col in [("Cases OOS", "oos"), ("Cases Unshipped", "unshipped"), ("Pick Omits", "pick_omits")]:
        per_month = g.groupby("month")[col].apply(lambda s: s.fillna(0).sum()).reindex(range(202601, 202609), fill_value=0)
        print(f"  {label} by month vs the deck's unlabeled monthly row: "
              f"{np.abs(per_month.to_numpy() - deck_months).sum():.1f} cases apart over 8 months")
else:
    print("The deck shows no figure for this distributor.")

**What it shows**

- The deck has no figure for Distributor B.
- With the deck's formula, Distributor B fills **94.3%** over the whole file and **93.0% for January–August 2026**, below the 97% target.
- Column N equals the formula within half a point on 7,527 of 7,562 rows. Part 1 found it tracks shipped / ordered; both hold because Unshipped = ordered − shipped on this file (section 8).
- **Reading:** Distributor A and Distributor B compute column N the same way. Distributor A only differs by leaving Unshipped blank on some rows.

### 8. Column identities, in cases

**Question.** How do the quantity columns fit together, and how many cases fall outside?

The deck's sample rows (slide 8) suggest: Cases OOS = ordered − confirmed, Pick Omits = confirmed − shipped, and Cases Unshipped = OOS + Pick Omits = ordered − shipped.

**How to read the output.** Each identity is tested on every row, with blanks as zero. "Rows holding" counts rows where both sides agree within 0.01 case. "Cases apart" adds up the gap on the rows that do not hold. The next tables split those rows by which cells are blank. On Distributor A, a last block sorts the blank Cases Unshipped cells.

In [ ]:
# Candidate identities, read from the sample rows on slide 8 of the deck. Tested on every row, in cases.
# Blank cells count as 0 in this cell. A row holds when both sides are within 0.01 case.
z = df[["ordered", "confirmed", "shipped", "unshipped", "pick_omits", "oos"]].fillna(0)
identities = [
    ("Cases OOS = ordered - confirmed", z["oos"], z["ordered"] - z["confirmed"], ["oos"]),
    ("Pick Omits = confirmed - shipped", z["pick_omits"], z["confirmed"] - z["shipped"], ["pick_omits", "shipped"]),
    ("Unshipped = ordered - shipped", z["unshipped"], z["ordered"] - z["shipped"], ["unshipped", "shipped"]),
    ("Unshipped = OOS + Pick Omits", z["unshipped"], z["oos"] + z["pick_omits"], ["unshipped", "oos", "pick_omits"]),
]
summary = []
for label, left, right, _cols in identities:
    holds = (left - right).abs() <= 0.01
    summary.append({
        "identity": label,
        "rows holding": f"{int(holds.sum()):,} of {len(df):,}",
        "left side, cases": cases(left.sum()),
        "right side, cases": cases(right.sum()),
        "cases apart on failing rows": cases((left - right)[~holds].abs().sum()),
    })
print(pd.DataFrame(summary).to_string(index=False))

def blank_pattern(cols):
    flags = df[cols].isna()
    return flags.apply(lambda r: "blank: " + ", ".join(c for c in cols if r[c]) if r.any() else "no blank", axis=1)

for label, left, right, cols in identities:
    diff = (left - right).abs()
    off = diff > 0.01
    if not off.any():
        continue
    where = diff[off].groupby(blank_pattern(cols)[off]).agg(["size", "sum"])
    where.columns = ["failing rows", "cases apart"]
    where["cases apart"] = where["cases apart"].map(cases)
    print(f"\n{label}: failing rows by blank cells")
    print(where.to_string())

if df["unshipped"].isna().any():
    blank = df["unshipped"].isna()
    gap = df["ordered"] - df["shipped"].fillna(0)
    fits_zero = blank & df["shipped"].notna() & (gap <= 0.05)
    print(f"\nBlank Cases Unshipped: {int(blank.sum()):,} rows")
    print(f"  shipped equals ordered, so a zero fits: {int(fits_zero.sum()):,} rows")
    for label, mask in [("shipped below ordered", blank & df["shipped"].notna() & (gap > 0.05)),
                        ("shipped blank too", blank & df["shipped"].isna())]:
        print(f"  {label}: {int(mask.sum()):,} rows, {cases(gap[mask].sum())} cases ordered and not shipped, "
              f"{cases(df.loc[mask, 'short'].sum())} of them never confirmed")
    as_miss = unshipped0.where(~blank, gap.clip(lower=0))
    for label, mask in [("whole file", all_rows), ("last 12 months", last12), ("Jan-Aug 2026", ytd)]:
        print(f"  Deck formula, {label}: {pct(deck_fill(mask))} with blanks as zero, "
              f"{pct(1 - as_miss[mask].sum() / df.loc[mask, 'ordered'].sum())} with blanks as ordered - shipped")

**What it shows**

- All four identities hold: Pick Omits = confirmed − shipped on 7,561 of 7,562 rows, Unshipped = ordered − shipped on 7,542, Unshipped = OOS + Pick Omits on 7,495, OOS = ordered − confirmed on 7,474.
- At most 293 cases fall outside an identity, and no cell is blank.
- **Reading:** on this file Cases OOS is the confirmation gap, Pick Omits the gap between confirmed and shipped, and Unshipped the two together.

### 9. Row average or case average

**Question.** Why does column N average about 88–90% over rows when 94–96% of the cases are confirmed?

**How to read the output.** Rows are cut into five equal groups by cases ordered (Q1 = smallest rows). The shares say how much of the rows, the cases and the missed cases each group holds. Then the same file is averaged four ways: by rows or by cases, with column N or with confirmed / ordered.

In [ ]:
# Quintiles of cases ordered per row. Rows are ranked first, so ties do not merge buckets.
size_bucket = pd.qcut(df["ordered"].rank(method="first"), 5,
                      labels=["Q1 smallest", "Q2", "Q3", "Q4", "Q5 largest"])
row_conf = 1 - df["short"] / df["ordered"]
q = (df.assign(bucket=size_bucket, row_conf=row_conf)
     .groupby("bucket", observed=True)
     .agg(low=("ordered", "min"), high=("ordered", "max"), rows=("ordered", "size"),
          ordered=("ordered", "sum"), short=("short", "sum"),
          mean_n=("fill", "mean"), mean_conf=("row_conf", "mean")))
table = pd.DataFrame({
    "cases per row": [f"{lo:,.2f} to {hi:,.2f}" for lo, hi in zip(q["low"], q["high"])],
    "rows": (q["rows"] / q["rows"].sum()).map(pct),
    "cases": (q["ordered"] / q["ordered"].sum()).map(pct),
    "missed cases": (q["short"] / q["short"].sum()).map(pct),
    "confirmation fill": (1 - q["short"] / q["ordered"]).map(pct),
    "row mean, col N": q["mean_n"].map(pct),
}, index=q.index)
print(table.to_string())

print("\nSame file, four ways to average")
print("  Row mean, column N                  ", pct(df["fill"].mean()))
print("  Row mean, confirmed / ordered       ", pct(row_conf.mean()))
print("  Weighted by cases, column N         ", pct((df["fill"] * df["ordered"]).sum() / df["ordered"].sum()))
print("  Weighted by cases, confirmed/ordered", pct(1 - df["short"].sum() / df["ordered"].sum()))

**What it shows**

- The largest fifth of rows carries **88.8% of the cases** and confirms 94.8%. The smallest fifth carries 0.3% of the cases and confirms 88.1%.
- Four averages of the same file: 90.3% (column N, by rows), 90.5% (confirmed / ordered, by rows), 94.3% (column N, by cases), 94.4% (confirmed / ordered, by cases). **Weighting moves the figure by 4 points; the choice of column by less than half a point.**

### 10. Rows that did not ship in full but score 100%

**Question.** Where are the cases that column N scores as fully served although less was shipped? And the cases confirmed with no shipped quantity?

**How to read the output.** One block per group: its size, the months it falls in (largest five shown), how it spreads over the row-size groups of section 9, and what the other columns say on those rows.

In [ ]:
if df["unshipped"].isna().any():
    hidden_rows = df["unshipped"].isna() & df["shipped"].notna() & (df["ordered"] - df["shipped"] > 0.05)
else:
    hidden_rows = (df["fill"] >= 0.999) & (df["ordered"] - df["shipped"] > 0.05)
no_ship_rows = df["shipped"].isna() & (df["confirmed"] > 0)
groups = [
    ("Scored 100% by column N, shipped below ordered", hidden_rows,
     (df["ordered"] - df["shipped"]).where(hidden_rows, 0)),
    ("Confirmed, no shipped quantity (ordered cases)", no_ship_rows, df["ordered"].where(no_ship_rows, 0)),
]
volume_by_month = df.groupby("month")["ordered"].sum()
for label, rows_mask, amount in groups:
    print(f"\n{label}")
    if not rows_mask.any():
        print("  No row.")
        continue
    total = amount.sum()
    print(f"  Rows {int(rows_mask.sum()):,} | cases {cases(total)} | {pct(total / df['ordered'].sum())} of all ordered cases")
    per_month = amount.groupby(df["month"]).sum()
    per_month = per_month[per_month > 0.05].sort_values(ascending=False)
    print(f"  Months with any: {len(per_month)} of {len(months)} | largest three months hold "
          f"{pct(per_month.head(3).sum() / total)}")
    for month, value in per_month.head(5).items():
        print(f"    {int(month)}  {cases(value):>6} cases  {pct(value / volume_by_month[month])} of that month's ordered cases")
    by_size = amount.groupby(size_bucket, observed=True).sum() / total
    print("  Share by row-size quintile, smallest first:", [pct(v) for v in by_size])
    print("  Median cases ordered on these rows:", f"{df.loc[rows_mask, 'ordered'].median():,.1f}",
          "| all rows:", f"{df['ordered'].median():,.1f}")
    print("  Rows fully confirmed:", pct((df.loc[rows_mask, "short"] <= 0.01).mean()),
          "| Pick Omits filled:", pct(df.loc[rows_mask, "pick_omits"].notna().mean()),
          "| Cases OOS filled:", pct(df.loc[rows_mask, "oos"].notna().mean()))
    holds = (unshipped0 - (df["ordered"] - df["shipped"].fillna(0))).abs() <= 0.01
    print("  Rows where Cases Unshipped = ordered - shipped, blanks as 0:", pct(holds[rows_mask].mean()))

**What it shows**

- Only 7 rows (9 cases) score 100% while shipping less than ordered: rounding on very large rows.
- No row has cases confirmed without a shipped quantity. Nothing to fix here.

### 11. One sales org or several

**Question.** When an id is missed in a month, is it missed in every sales org that ordered it, or only in some?

**How to read the output.** Each id-month with a miss falls in one of three groups. A miss in every org at once points to a cause they share; a miss in one org only points to a local one.

In [ ]:
# For each product id and month: how many sales orgs ordered it, and how many were short (> 0.05 case).
id_month = df.groupby(["month", "sku"]).agg(orgs_ordering=("org", "nunique"), short=("short", "sum"))
id_month["orgs_short"] = (df[df["short"] > 0.05].groupby(["month", "sku"])["org"].nunique()
                          .reindex(id_month.index, fill_value=0))
missed = id_month[id_month["orgs_short"] > 0]
kind = np.select(
    [missed["orgs_ordering"] == 1, missed["orgs_short"] == missed["orgs_ordering"]],
    ["only one org ordered it", "several orgs ordered it, all short"],
    default="several orgs ordered it, some short",
)
spread = missed.groupby(kind).agg(id_months=("short", "size"), short=("short", "sum"))
spread["share of id-months"] = (spread["id_months"] / spread["id_months"].sum()).map(pct)
spread["share of missed cases"] = (spread["short"] / spread["short"].sum()).map(pct)
print(spread[["share of id-months", "share of missed cases"]].to_string())
several = missed[missed["orgs_ordering"] > 1]
print("Where several orgs ordered the id, average share of them that were short:",
      pct((several["orgs_short"] / several["orgs_ordering"]).mean()))

**What it shows**

- **73% of missed cases** fall in id-months where both orgs ordered the id and both fell short.
- Caution before comparing with Distributor A: both of Distributor B's orgs are in the same state, and we do not know whether they ship from the same warehouse. If they do, missing in both at once is expected. The question is in `docs/open_questions.md`.

### 12. The weakest months

**Question.** In the months under 90% confirmation fill (or the weakest month, if none is under 90%), are the missed cases on the ids that miss all the time, or on other ids?

**How to read the output.** "Ids also short in 6 or more other months" is the chronic set, counted without the months examined, so the test is not circular. The volume lines compare the smaller and the larger half of the months. One line per month examined follows.

In [ ]:
low_months = list(monthly.index[monthly["wfill"] < 0.90]) or [monthly["wfill"].idxmin()]
in_low = df["month"].isin(low_months)
short_elsewhere = df[~in_low & (df["short"] > 0.05)].groupby("sku")["month"].nunique()
chronic_elsewhere = short_elsewhere[short_elsewhere >= 6].index
top10 = by_sku.head(10).index
low_by_sku = df[in_low].groupby("sku")["short"].sum().sort_values(ascending=False)
low_by_sku = low_by_sku[low_by_sku > 0.05]
low_total = low_by_sku.sum()
print("Months examined:", [int(m) for m in low_months], "(under 90%, or the weakest month if none)")
print("Missed cases in these months:", cases(low_total), f"({pct(low_total / df['short'].sum())} of the file's)",
      "| ids with a miss:", len(low_by_sku))
print("Share from ids also short in 6 or more other months:",
      pct(low_by_sku[low_by_sku.index.isin(chronic_elsewhere)].sum() / low_total))
print("Share from ids short in no other month:",
      pct(low_by_sku[~low_by_sku.index.isin(short_elsewhere.index)].sum() / low_total))
print("Share from ids short in 6 or more months, these months included (section 4):",
      pct(low_by_sku[low_by_sku.index.isin(chronic)].sum() / low_total))
print("Share from the 10 largest ids of the whole file:",
      pct(low_by_sku[low_by_sku.index.isin(top10)].sum() / low_total))
top5_each = [set(df[(df["month"] == m)].groupby("sku")["short"].sum().nlargest(5).index) for m in low_months]
repeat = set.union(*top5_each) if len(top5_each) > 1 else set()
repeat = {s for s in repeat if sum(s in t for t in top5_each) > 1}
print("Ids in the top 5 of more than one of these months:", len(repeat), "of", len(set().union(*top5_each)))
print("Ids carrying 80% of these months' missed cases:", int((low_by_sku.cumsum() / low_total < 0.8).sum()) + 1)
small_months = monthly["ordered"] <= monthly["ordered"].median()
for label, mask in [("Half of the months with the least volume", small_months),
                    ("Half of the months with the most volume ", ~small_months)]:
    part = monthly[mask]
    print(f"{label}: confirmation fill {pct(1 - part['short'].sum() / part['ordered'].sum())}, "
          f"months under 90%: {int((part['wfill'] < 0.90).sum())}")
print("Examined months that are among the smallest half by volume:",
      int(small_months[low_months].sum()), "of", len(low_months))
for month in sorted(low_months):
    g = df[(df["month"] == month) & (df["short"] > 0.05)]
    per_id = g.groupby("sku")["short"].sum().sort_values(ascending=False)
    print(f"  {int(month)}  missed {cases(per_id.sum())}  ids {len(per_id)}  "
          f"largest id {pct(per_id.iloc[0] / per_id.sum())}  largest 5 {pct(per_id.head(5).sum() / per_id.sum())}  "
          f"chronic elsewhere {pct(per_id[per_id.index.isin(chronic_elsewhere)].sum() / per_id.sum())}  "
          f"zero-confirm {pct(g.loc[g['confirmed'] == 0, 'short'].sum() / per_id.sum())}")

**What it shows**

- The four months hold **7,509 missed cases, 36% of the file's**, on 146 ids. Nine ids carry 80% of it; in each month the five largest ids hold 75% to 86%.
- **March 2025 is the chronic set:** 78% of its missed cases come from ids short in six or more other months.
- **July 2025, January 2026 and February 2026 are another set:** only 10% to 12% from those ids. A few large ids, 6 of which recur among these months' top five. February 2026's largest id alone is 67% of that month's 2,590 missed cases.
- **All four are small months:** the smaller half of the months confirms 90.7%, the larger half 96.2%, and all four months under 90% are in the smaller half.

### 13. Persistence

**Question.** Is a missed case usually on an id that was already short the month before?

**How to read the output.** Each missed case from the second month on is classed by what happened to the same id in the same sales org the month before. Two more lines widen this to any org, and to long runs.

In [ ]:
# Months follow each other with no gap, so the previous month is the previous position.
position = {m: i for i, m in enumerate(months)}
pos = df["month"].map(position)
was_short_keys = set(zip(pos[df["short"] > 0.05] + 1, df.loc[df["short"] > 0.05, "org"], df.loc[df["short"] > 0.05, "sku"]))
was_ordered_keys = set(zip(pos + 1, df["org"], df["sku"]))
now = df[(df["short"] > 0.05) & (pos > 0)].sort_values("month")
now_keys = list(zip(pos[now.index], now["org"], now["sku"]))
prev_state = np.where([k in was_short_keys for k in now_keys], "short the month before",
                      np.where([k in was_ordered_keys for k in now_keys], "ordered and filled the month before",
                               "not ordered the month before"))
persist = now.groupby(prev_state)["short"].sum()
print(f"Missed cases from the second month on: {cases(now['short'].sum())}. Same id in the same sales org:")
for label, value in persist.sort_values(ascending=False).items():
    print(f"  {label:38} {pct(value / now['short'].sum())}")

id_short_prev = set(zip(pos[df["short"] > 0.05] + 1, df.loc[df["short"] > 0.05, "sku"]))
any_org = np.array([(p, s) in id_short_prev for p, s in zip(pos[now.index], now["sku"])])
print("Share from ids short the month before in any sales org:", pct(now.loc[any_org, "short"].sum() / now["short"].sum()))

streak = {}
run_length = []
for (p, o, s) in zip(pos[now.index], now["org"], now["sku"]):
    run = streak.get((p - 1, o, s), 1) + 1 if (p, o, s) in was_short_keys else 1
    streak[(p, o, s)] = run
    run_length.append(run)
run_length = np.array(run_length)
print("Share in the third or later consecutive short month (same id, same org):",
      pct(now.loc[run_length >= 3, "short"].sum() / now["short"].sum()))

**What it shows**

- From the second month on, 46% of missed cases hit an id already short in the same org the month before, 33% an id ordered and filled, and **21% an id that org had not ordered** (often a new id, section 14).
- Any org: 50%. Third or later consecutive short month: 10.5%. Misses persist less than on Distributor A.

### 14. New product ids

**Question.** Do ids that appear during the period miss more than ids present from the start? Are they new products, or known products under a new id?

**How to read the output.** Two cohorts by first month on file, then the new ids' first three months against the rest of their life. The next lines test whether new ids reuse an earlier product name (a sign of a new vintage or label), when they appear, and how long they last. The last line looks at ids that stop.

In [ ]:
first_seen = df.groupby("sku")["month"].min()
last_seen = df.groupby("sku")["month"].max()
seen_from_start = df["sku"].map(first_seen) == months[0]
cohorts = [
    ("ids present in the first month", seen_from_start),
    ("ids first seen later", ~seen_from_start),
]
for label, mask in cohorts:
    print(f"{label:32} ids {df.loc[mask, 'sku'].nunique():4}  ordered {pct(df.loc[mask, 'ordered'].sum() / df['ordered'].sum())}  "
          f"missed {pct(df.loc[mask, 'short'].sum() / df['short'].sum())}  "
          f"confirmation fill {pct(1 - df.loc[mask, 'short'].sum() / df.loc[mask, 'ordered'].sum())}")

age = pos - df["sku"].map(first_seen).map(position)
new_early = ~seen_from_start & (age < 3)
new_later = ~seen_from_start & (age >= 3)
for label, mask in [("new ids, first 3 months on file", new_early), ("new ids, after that", new_later)]:
    print(f"  {label:32} ordered {cases(df.loc[mask, 'ordered'].sum()):>9}  "
          f"confirmation fill {pct(1 - df.loc[mask, 'short'].sum() / df.loc[mask, 'ordered'].sum())}")

# A later id could be a known product under a new id (new vintage, new label). Check the product name.
later_ids = first_seen[~first_seen.index.isin(df.loc[seen_from_start, "sku"])].index
names_by_id = df.groupby("sku")["product"].agg(set)
known_name = pd.Series({
    sku: any(names_by_id[sku] & names_by_id[e] for e in first_seen[first_seen < first_seen[sku]].index)
    for sku in later_ids
})
print("\nLater ids whose product name was already used by an earlier id:", int(known_name.sum()), "of", len(later_ids),
      "| share of the later ids' missed cases:",
      pct(df.loc[df["sku"].isin(known_name[known_name].index), "short"].sum() / df.loc[~seen_from_start, "short"].sum()))
brands_at_start = set(df.loc[df["month"] == months[0], "brand"])
print("Later ids in a brand already on file in the first month:",
      pct(df[~seen_from_start].groupby("sku")["brand"].first().isin(brands_at_start).mean()))
first_calendar_month = (first_seen[later_ids] % 100).value_counts().sort_index()
print("Later ids by calendar month of first appearance (Jan = 1):", {int(k): int(v) for k, v in first_calendar_month.items()})
months_on_file = df.groupby("sku")["month"].nunique()
brief = later_ids[months_on_file[later_ids] <= 3]
print("Later ids with rows in 3 months or fewer:", len(brief), "of", len(later_ids),
      "| share of missed cases:", pct(df.loc[df["sku"].isin(brief), "short"].sum() / df["short"].sum()))

gone = df["sku"].map(last_seen) < months[-3]
print("Ids with no row in the last 3 months:", int((last_seen < months[-3]).sum()),
      "| share of missed cases", pct(df.loc[gone, "short"].sum() / df["short"].sum()),
      "| share of ordered", pct(df.loc[gone, "ordered"].sum() / df["ordered"].sum()))

**What it shows**

- The 175 ids present in the first month carry 81% of volume and 56% of missed cases (96.1% confirmed). **The 216 ids first seen later carry 19% of volume and 44% of missed cases** (87.0%), and confirm only **79.1% in their first three months** (93.3% after).
- They are new products: none reuses an earlier product name, and 98.6% belong to a brand already on file. Short-lived ids (three months or fewer) carry only 2.6% of missed cases.
- Weak first months on new items look like a real driver here. One plausible reading, to test with the Planning files: a new item starts with little demand history.
- The 183 ids with no row in the last three months carry 21% of missed cases.

### 15. Concentration

**Question.** How many ids would you need to look at to cover most of the missed cases?

**How to read the output.** Shares of missed cases and of volume for the 10, 20, 50 and 100 largest ids. The chart shows the full curve; the horizontal axis is on a log scale so the first ids are readable.

In [ ]:
ranked = by_sku[by_sku > 0.05]
cumulative = ranked.cumsum() / ranked.sum()
volume_by_sku = df.groupby("sku")["ordered"].sum()
for n in [10, 20, 50, 100]:
    print(f"Largest {n:3} ids: {pct(cumulative.iloc[n - 1])} of missed cases, "
          f"{pct(volume_by_sku[ranked.index[:n]].sum() / volume_by_sku.sum())} of ordered cases")
for target in [0.5, 0.8]:
    print(f"Ids needed for {target:.0%} of missed cases:", int((cumulative < target).sum()) + 1, "of", len(ranked), "with a miss")

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(np.arange(1, len(cumulative) + 1), cumulative.to_numpy())
for n in [10, 20, 50, 100]:
    ax.plot(n, cumulative.iloc[n - 1], marker="o", ms=3.5, color="C0")
    ax.annotate(f"top {n}: {cumulative.iloc[n - 1]:.0%}", (n, cumulative.iloc[n - 1]),
                textcoords="offset points", xytext=(4, -12), fontsize=8)
ax.set_xscale("log")
ax.set_xticks([1, 10, 20, 50, 100, len(cumulative)])
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda value, _pos: f"{value:,.0f}"))
ax.xaxis.set_minor_formatter(plt.NullFormatter())
ax.set_ylim(0, 1.01)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _pos: f"{value:.0%}"))
ax.set_xlabel("Product ids, largest miss first (log scale)")
ax.set_ylabel("Share of missed cases")
ax.set_title(f"Distributor {DISTRIBUTOR.upper()} — concentration of missed cases by product id")
fig.tight_layout()
plt.show()

**What it shows**

- The 10, 20, 50 and 100 largest ids carry 56%, 69%, 84% and 93% of missed cases. **8 ids make half**, 38 make 80%.
- The top 10 carry 46% of the volume and 56% of missed cases: here the biggest ids miss more than their weight. A short list of ids covers most of the gap.

### 16. Unit: physical or 9-liter cases

**Question.** Are the fractional cases physical cases (bottles out of a pack) or 9-liter equivalent cases? The deck does not say.

**How to read the output.** The pack count ("6B") and bottle size ("750ML") are read from the product name, which is parsed and never printed. For each fractional quantity, two readings are tried: as physical cases, ordered × pack must be a whole number of bottles; as 9-liter cases, ordered × 9 / bottle size must be. The line that decides is the one on formats where the two readings disagree. This is evidence for the team. Nothing is converted.

In [ ]:
# Pack count ("6B") and bottle size ("750ML", "1.75L") are read from the product name. Names are never printed.
names = df["product"].astype(str)
pack = pd.to_numeric(names.str.extract(r"\b(\d{1,2})B\b")[0], errors="coerce")
size = names.str.extract(r"\b(\d+(?:\.\d+)?)\s?(ML|L)\b", flags=re.IGNORECASE)
litres = pd.to_numeric(size[0], errors="coerce") / np.where(size[1].str.upper() == "ML", 1000, 1)
# Keep sizes where both readings give fewer than 30 bottles per case, so a match is not luck.
usable = pack.notna() & litres.notna() & (pack <= 24) & (litres >= 0.3)
fractional = usable & ((df["ordered"] - df["ordered"].round()).abs() > 0.01)
print("Rows with a pack count and bottle size in the name:", pct(usable.mean()),
      "| of which with a fractional quantity:", f"{int(fractional.sum()):,}")

def whole(values):
    return (values - values.round()).abs() <= 0.05

bottles_if_physical = df["ordered"] * pack
bottles_if_nine_litre = df["ordered"] * 9 / litres
test = pd.DataFrame({
    "pack": pack, "litres": litres,
    "physical": whole(bottles_if_physical), "nine_litre": whole(bottles_if_nine_litre),
})[fractional]
print("Fractional quantities that are whole bottles if a case is a physical case:", pct(test["physical"].mean()))
print("Fractional quantities that are whole bottles if a case is a 9-liter case: ", pct(test["nine_litre"].mean()))
# For 6 x 750 ml, a whole number of physical bottles is also a whole number under the 9-liter reading.
# The formats below are the ones where the two readings disagree.
ratio = 9 / test["litres"] / test["pack"]
differ = (ratio - ratio.round()).abs() > 0.01
print(f"Formats where the readings disagree ({int(differ.sum()):,} rows): physical {pct(test.loc[differ, 'physical'].mean())}, "
      f"9-liter {pct(test.loc[differ, 'nine_litre'].mean())}")
by_format = test.groupby(["pack", "litres"]).agg(rows=("physical", "size"), physical=("physical", "mean"),
                                                 nine_litre=("nine_litre", "mean"))
by_format = by_format.sort_values("rows", ascending=False).head(8)
by_format["physical"] = by_format["physical"].map(pct)
by_format["nine_litre"] = by_format["nine_litre"].map(pct)
print(by_format.to_string())

**What it shows**

- 2,432 fractional quantities can be tested (pack and size found in the name for 59% of rows).
- On formats where the two readings disagree (291 rows): **100% are whole bottles as physical cases, 35% as 9-liter cases.**
- The quantities read as physical cases. On 2026-10-23 the client confirmed that reading. The 9L factor stays a separate column.

### 17. Keys and scope

**Question.** Can the product name serve as a key? What are the duplicated keys, if any? Which sales orgs share a state?

**How to read the output.** Counts only. "Side by side" means two ids (or two names) present in the same month and the same sales org.

In [ ]:
# Product ids and product names over time. Counts only.
def overlap_count(groups, inner):
    """Groups whose members (ids or names) appear in the same month and sales org."""
    concurrent = 0
    for _, g in groups:
        if g.groupby(["month", "org"])[inner].nunique().max() > 1:
            concurrent += 1
    return concurrent

multi_id_names = df[df["product"].map(df.groupby("product")["sku"].nunique()) > 1]
multi_name_ids = df[df["sku"].map(df.groupby("sku")["product"].nunique()) > 1]
print("Names with several ids:", multi_id_names["product"].nunique(),
      "| of which the ids sit side by side in the same month and org:",
      overlap_count(multi_id_names.groupby("product"), "sku"))
print("Ids with several names:", multi_name_ids["sku"].nunique(),
      "| of which the names sit side by side in the same month and org:",
      overlap_count(multi_name_ids.groupby("sku"), "product"))

dups = df[df.duplicated(keys, keep=False)]
if len(dups):
    pair_names = dups.groupby(keys)["product"].agg(lambda s: sorted(set(s)))
    two = pair_names[pair_names.map(len) == 2]

    def token_change(pair):
        a, b = (set(n.split()) for n in pair)
        changed = a ^ b
        if changed and all(re.fullmatch(r"(19|20)\d{2}", t) for t in changed):
            return "only a year changes"
        if changed and all(re.fullmatch(r"[\d.]+\w{0,2}", t) for t in changed):
            return "only a number changes"
        return "words change"

    print("\nDuplicated month x org x id keys:", len(pair_names), "| with exactly two names:", len(two))
    print("  How the two names differ:", pd.Series([token_change(p) for p in two]).value_counts().to_dict())
    span = df.groupby(["sku", "product"])["month"].agg(["min", "max"])
    handover = 0
    for (month, _org, sku), (name_1, name_2) in two.items():
        a, b = span.loc[(sku, name_1)], span.loc[(sku, name_2)]
        if (a["max"] == month and b["min"] == month) or (b["max"] == month and a["min"] == month):
            handover += 1
    print("  Keys where the month is the last month of one name and the first of the other:", handover)
    print("  Ordered cases on these rows:", cases(dups["ordered"].sum()),
          "| missed cases:", cases(dups["short"].sum()))

# Sales orgs that share a state. Printed by rank, never by name.
orgs_per_state = df.groupby("state")["org"].nunique()
for state in orgs_per_state[orgs_per_state > 1].index:
    in_state = df[df["state"] == state]
    print("\nState with", in_state["org"].nunique(), "sales orgs")
    state_ids = {o: set(g["sku"]) for o, g in in_state.groupby("org")}
    ranked_orgs = in_state.groupby("org")["ordered"].sum().sort_values(ascending=False)
    for rank, org_name in enumerate(ranked_orgs.index, start=1):
        g = in_state[in_state["org"] == org_name]
        others = set().union(*(ids for o, ids in state_ids.items() if o != org_name))
        print(f"  org #{rank}: months {int(g['month'].min())}-{int(g['month'].max())} ({g['month'].nunique()}), "
              f"ordered {pct(g['ordered'].sum() / df['ordered'].sum())} of file, "
              f"missed {pct(g['short'].sum() / df['short'].sum())} of file, "
              f"ids also sold by the other org {len(state_ids[org_name] & others)} of {len(state_ids[org_name])}")

**What it shows**

- 11 ids carry several names. For 6 of them, two names sit side by side in the same month and org: these are the duplicated keys.
- 11 of the 14 duplicated keys fall in the month where one name hands over to the other, and 2 differ only by a year. Each looks like one id-month split across a name change. They hold 241 ordered cases and 36 missed cases.
- Both sales orgs have rows in all 25 months and share 364 ids.

### 18. Checks, second pass

**What happens.** The new sections only read. This cell checks that the published totals have not moved and, on Distributor A, pins the deck's 95.7%.

In [ ]:
# The second pass only reads. The published totals must not move.
assert list(df.columns) == [
    "month", "state", "org", "vendor", "brand", "product", "sku",
    "ordered", "confirmed", "shipped", "unshipped", "pick_omits", "oos", "fill", "short",
]
if DISTRIBUTOR == "a":
    assert len(df) == 29118
    assert abs(df["ordered"].sum() - 1794811) < 1 and abs(df["short"].sum() - 67812.6) < 1
    assert abs(deck_fill(ytd) - 0.957) < 0.0005  # "currently 95.7%" on slide 10 of the deck
elif DISTRIBUTOR == "b":
    assert len(df) == 7562
    assert abs(df["ordered"].sum() - 373828) < 1 and abs(df["short"].sum() - 21110) < 1
else:
    raise AssertionError(DISTRIBUTOR)
print("Checks passed. Published totals unchanged.")

## Part 3 — Before any cleaning

Nothing above changes the file. The choices below would. They are proposals for the team meeting, tracked in `docs/open_questions.md` (team questions 4 to 8). None is applied here.

### Proposals for the team meeting

1. **Fill-rate definition.** Decided 2026-10-23. Use the deck's: 1 − Σ Cases Unshipped / Σ Cases Ordered, summed over cases, for both distributors. On this file it equals shipped / ordered within a few cases. Never average column N over rows. (Sections 7, 9.)
2. **Unit.** Decided 2026-10-23: keep the cases as physical cases. The 9L factor is joined in the clean working file, not in this notebook. (Section 16.)
3. **Key.** Decided 2026-10-23: column G, not the product name. The 14 duplicated keys are summed in the clean working file. On this distributor column G is the distributor item code, not the client SAP SKU. (Section 17.)

### What this notebook does not say

The forecast, ADU, buffers and lead times are in the Planning files, not in this workbook, and the deck defines no link to Master Data or Planning, so no join was tried. This notebook shows where cases were missed. It does not say whether a buffer was too small, nor why new products start badly.

Update 2026-10-26: the choices the client answered are applied in the clean working file, and Part 4 reads that file. Distributor A's blank unshipped cells are still open, so that file is not cleaned here.

Update 2026-11-04: the frozen template keeps the 14 duplicate keys as separate rows. `distributor_b_v1.xlsx` does that. The October clean file still has them summed.


### Open questions

Status in `docs/open_questions.md` after reading the client's deck:

| Question | Status |
|---|---|
| Team 5. Which fill-rate definition for both distributors? | **Answered, 2026-10-23.** Case-weighted: 1 − Σ unshipped / Σ ordered. The row average is secondary. |
| Team 6. Physical or 9-liter cases? | **Answered, 2026-10-23.** Physical cases. The 9L factor is separate. |
| Team 7. Is column G the aggregation key, including for the 14 duplicated keys? | **Answered, 2026-10-23.** Yes for the within-file grain, with the duplicated rows summed. It is the distributor item code, not the client SAP SKU. |
| Client 11. Do Distributor B's two sales orgs ship from the same warehouse? | **Partial, 2026-10-23.** The location list maps both names to one market. It does not say they share a warehouse. |

## Part 4 — Where the lost cases are

Written on 2026-10-26 from the clean working file (`data/cleaned/distributor_b_clean.xlsx`). That file is local only. This part reads it and writes nothing.

The rules are the ones agreed on 2026-10-23. One row of the ranking is one source id in the one market: the two sales orgs are added, because the location list gives them one market. A shared warehouse is still unconfirmed, so the addition is a view. The clean grain stays sales org x source id x month.

The main cut leaves out the three flagged RTS ids. Ids with no 9L factor stay off the ranking. They are not treated as zero loss. Every display type stays in. A second cut keeps naked goods and gift box only.

Lost volume is unshipped cases times the 9L factor. Fill rate is 1 − unshipped / ordered, summed over cases. Frequent means unshipped cases above 0.01 in 4 or more distinct months. Large means the id is inside the Pareto, including the row that crosses 80% of unshipped 9L. The quadrant names are the readings from the client calls. They are not root causes: this extract has no forecast, no lead time and no buffer.

The same cuts are formulas in the workbook, on Analysis, Pareto, Pareto_NG and Seasonality. The numbers below are computed again here from the stored cases and the 9L factor, so they can be checked without opening Excel.


### 19. The cut

**Question.** Once the agreed rules are applied, how large is the loss we are about to rank?

**How to read the output.** Physical cases and 9L, for the whole file, for the three RTS ids, and for the ids that have a factor. Fill rate is summed over cases. The RTS ids are counted, not printed.


In [ ]:
clean_path = Path("data/cleaned") / "distributor_b_clean.xlsx"
if not clean_path.exists():
    raise FileNotFoundError("The clean working file is local only. It is not in git.")

clean = pd.read_excel(clean_path, sheet_name="Clean")
rts_ids = set(
    pd.to_numeric(pd.read_excel(clean_path, sheet_name="RTS_Ids").iloc[:, 0], errors="coerce")
    .dropna()
    .astype(int)
)
if len(rts_ids) != 3:
    raise RuntimeError(f"Expected 3 flagged RTS ids, found {len(rts_ids)}")

clean["month"] = pd.to_datetime(clean["Month"])
clean["factor"] = pd.to_numeric(clean["Nine_L_Factor"], errors="coerce")
clean["ordered"] = clean["Cases_Ordered"]
clean["unshipped"] = clean["Cases_Unshipped"]
clean["ord9"] = clean["ordered"] * clean["factor"]
clean["uns9"] = clean["unshipped"] * clean["factor"]
clean["rts"] = clean["Source_Id"].isin(rts_ids)
clean["display"] = clean["Display_Type"].fillna("(blank)")

if abs(clean["ordered"].sum() - 373_828.151) > 0.001:
    raise RuntimeError("Clean ordered total does not reconcile to the raw file")
if abs(clean["unshipped"].sum() - 21_325.637) > 0.001:
    raise RuntimeError("Clean unshipped total does not reconcile to the raw file")

def block(mask, label):
    part = clean.loc[mask]
    ordered = part["ordered"].sum()
    unshipped = part["unshipped"].sum()
    ordered_9l = part["ord9"].sum()
    unshipped_9l = part["uns9"].sum()
    print(
        f"{label:22} rows {len(part):5}  ids {part['Source_Id'].nunique():4}  "
        f"ordered {cases(ordered):>10}  unshipped {cases(unshipped):>8}  fill {pct(1 - unshipped / ordered):>6}  "
        f"unshipped 9L {cases(unshipped_9l):>8}  fill 9L {pct(1 - unshipped_9l / ordered_9l) if ordered_9l else 'n/a':>6}"
    )

print(f"Clean rows {len(clean):,} | months {clean['month'].nunique()} | sales orgs {clean['Sales_Org'].nunique()} | markets {clean['Market'].nunique()}")
print(f"Flagged RTS ids: {len(rts_ids)}. Not printed.")
block(slice(None), "All rows")
block(clean["rts"], "RTS only")
block(~clean["rts"], "RTS excluded")
block(~clean["rts"] & clean["factor"].notna(), "Main cut")
block(clean["factor"].isna(), "No 9L factor")


**What it shows**

- The clean file reconciles: 373,828 cases ordered, 21,326 unshipped, fill **94.3%** over August 2024 to August 2026.
- Leaving out the three RTS ids raises that to **94.8%** in physical cases and **95.1%** in 9L. Their own fill is **81.4%** (2,488 cases, 1,244 in 9L). Excluding them does not reach the 97% target named on the client's deck.
- The ranking below uses the main cut: 368 ids that have a 9L factor. The 20 ids with no factor have 160 unshipped cases, under 1% of the RTS-excluded physical loss. They stay on Exceptions. They are not in the Pareto, and they are not scored as a full fill.


### 20. Where it concentrates

**Question.** How many source ids carry 80% of the unshipped 9L?

**How to read the output.** Ids are ranked by unshipped 9L, RTS excluded. Top includes the id that crosses 80%. The chart has no id labels: the horizontal line is 80%, and the vertical line is the last Top id. The second table repeats the ranking on naked goods and gift box only. Display type is a category, not a product name.


In [ ]:
LOSS_FLOOR = 0.01
FREQUENT_MONTHS = 4
LARGE_SHARE = 0.8

base = clean.loc[~clean["rts"] & clean["factor"].notna()].copy()
id_month = base.groupby(["Source_Id", "month"], as_index=False).agg(
    ordered=("ordered", "sum"),
    unshipped=("unshipped", "sum"),
    uns9=("uns9", "sum"),
    display=("display", "first"),
    first_month=("month", "min"),
)
ranked = id_month.groupby("Source_Id", as_index=False).agg(
    months=("month", "nunique"),
    loss_months=("unshipped", lambda values: int((values > LOSS_FLOOR).sum())),
    ordered=("ordered", "sum"),
    unshipped=("unshipped", "sum"),
    uns9=("uns9", "sum"),
    display=("display", "first"),
    first_month=("month", "min"),
)
ranked = ranked.sort_values(["uns9", "Source_Id"], ascending=[False, True]).reset_index(drop=True)
total_uns9 = ranked["uns9"].sum()
ranked["previous_share"] = ranked["uns9"].cumsum().shift(fill_value=0) / total_uns9
ranked["top"] = (ranked["uns9"] > 0) & (ranked["previous_share"] < LARGE_SHARE)
ranked["frequent"] = ranked["loss_months"] >= FREQUENT_MONTHS
ranked["chronic"] = ranked["loss_months"] >= 6

def quadrant(row):
    if row["uns9"] <= 0:
        return "NO_LOSS"
    if row["top"] and row["frequent"]:
        return "FREQ_LARGE"
    if row["top"]:
        return "RARE_LARGE"
    if row["frequent"]:
        return "FREQ_REST"
    return "RARE_REST"

ranked["quadrant"] = ranked.apply(quadrant, axis=1)

def summarise(frame, label):
    ordered = frame.sort_values(["uns9", "Source_Id"], ascending=[False, True]).reset_index(drop=True)
    total = ordered["uns9"].sum()
    previous = ordered["uns9"].cumsum().shift(fill_value=0) / total
    top = (ordered["uns9"] > 0) & (previous < LARGE_SHARE)
    print(
        f"{label}: {int(top.sum())} of {len(ordered)} ids, "
        f"{pct(top.mean())} of ids, {pct(ordered.loc[top, 'uns9'].sum() / total)} of unshipped 9L, "
        f"{pct(ordered.loc[top, 'unshipped'].sum() / ordered['unshipped'].sum())} of physical unshipped"
    )
    return ordered, top

print("All display types")
summarise(ranked, "  main cut")
naked_gift = ranked[ranked["display"].isin(["NAKED GOODS", "GIFT BOX"])].copy()
print("Naked goods and gift box only")
summarise(naked_gift, "  subset")

twenty = max(1, round(0.20 * len(ranked)))
print(f"First 20% of ids ({twenty}) carry {pct(ranked.head(twenty)['uns9'].sum() / total_uns9)} of unshipped 9L")
print(f"Ids with any loss: {(ranked['loss_months'] > 0).sum()} of {len(ranked)}")

view = ranked.loc[ranked["top"], ["months", "loss_months", "unshipped", "uns9", "display", "quadrant"]].copy()
view.insert(0, "rank", range(1, len(view) + 1))
view["share"] = view["uns9"] / total_uns9
print("\nTop ids, no source id printed")
print(view.head(10).round({"unshipped": 0, "uns9": 0, "share": 3}).to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 3.6))
share = ranked["uns9"].cumsum() / total_uns9
ax.plot(range(1, len(share) + 1), share.to_numpy(), color="0.15")
ax.axhline(LARGE_SHARE, color="0.45", lw=1)
ax.axvline(int(ranked["top"].sum()), color="0.45", lw=1, ls="--")
ax.set_ylim(0, 1.02)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _pos: f"{value:.0%}"))
ax.set_xlabel("Source ids, ranked by unshipped 9L")
ax.set_ylabel("Share of unshipped 9L")
ax.set_title(f"Distributor {DISTRIBUTOR.upper()}: concentration of unshipped 9L, RTS excluded")
fig.tight_layout()
plt.show()


**What it shows**

- **37 of 368 ids (10%) carry 80% of the unshipped 9L.** The client's rule of thumb was about 20% of cells and about 80% of the loss. On this market the loss is tighter than that: the first 20% of ids carry about 91%.
- Naked goods and gift box, taken alone, tell the same story: **32 of 313 ids carry 80%** of that subset.
- Ranking on physical cases instead of 9L would move a few ids across the line. 9L is the ranking the client asked for, because pack sizes then compare. The workbook keeps both columns.
- Display type is not the whole explanation, and it is not noise. Naked goods are a 96.6% fill and a bit over half of the unshipped 9L. Gift box is an 85.9% fill and about 29% of the unshipped 9L, on much less volume. VAP fills about 70% and limited edition about 80%, and together they are about 16% of the unshipped 9L.


### 21. Frequent or rare

**Question.** Do the ids that carry the loss miss in many months, or in a few?

**How to read the output.** Frequent means 4 or more months above the loss floor. That is the repeat rule in the format reference, and on this file it is also about the upper quarter of ids that miss at least once. Chronic means 6 or more months, which is the word this notebook used during exploration. Large means the Pareto top from section 20. `FREQ_LARGE` is the buffer or variability reading. `RARE_LARGE` is the supply or allocation reading. Neither is a cause.


In [ ]:
def matrix(frame, top_mask, frequent_mask, label):
    print(label)
    rows = []
    for name, mask in [
        ("FREQ_LARGE", top_mask & frequent_mask),
        ("RARE_LARGE", top_mask & ~frequent_mask & (frame["uns9"] > 0)),
        ("FREQ_REST", ~top_mask & frequent_mask),
        ("other", ~top_mask & ~frequent_mask),
    ]:
        uns9 = frame.loc[mask, "uns9"].sum()
        rows.append((name, int(mask.sum()), uns9, uns9 / frame["uns9"].sum()))
    out = pd.DataFrame(rows, columns=["quadrant", "ids", "uns9", "share"])
    print(out.round({"uns9": 0, "share": 3}).to_string(index=False))

print("All display types, 4-month rule")
counts = ranked["quadrant"].value_counts()
shares = ranked.groupby("quadrant")["uns9"].sum() / total_uns9
quad = pd.DataFrame({"ids": counts, "share_of_9L": shares}).reindex(
    ["FREQ_LARGE", "RARE_LARGE", "FREQ_REST", "RARE_REST", "NO_LOSS"]
)
print(quad.round(3).to_string())

print("\nSame top ids, 6-month rule")
matrix(ranked, ranked["top"], ranked["chronic"], "  chronic")

ng_order, ng_top = summarise(naked_gift, "naked and gift")
ng_frequent = ng_order["loss_months"] >= FREQUENT_MONTHS
print()
matrix(ng_order, ng_top, ng_frequent, "Naked and gift, 4-month rule")

new_top = ranked["top"] & (ranked["first_month"] > "2024-08-01")
print(
    f"\nTop ids absent from the first month: {int(new_top.sum())} of {int(ranked['top'].sum())}, "
    f"{pct(ranked.loc[new_top, 'uns9'].sum() / total_uns9)} of main-cut unshipped 9L"
)
print("Top ids with a deletion flag or IG Active = No are counted in the workbook. Both counts are 0.")


**What it shows**

- Under the 4-month rule, **32 ids are frequent and large. They hold 76% of the main-cut unshipped 9L.** Five ids are large and rarer. They hold 4%. On this cut, the concentrated loss is mostly repeated.
- Under the 6-month rule the same top splits: **22 ids hold 48%, and 15 hold 32%.** Calling 4 months "frequent" makes the loss look like one repeated problem. Calling 6 months "frequent" leaves a large middle. Both numbers are in the workbook. The label is not a cause until replenishment looks at buffers and supply.
- Naked goods and gift box match the 4-month reading: 28 frequent-and-large ids hold 77% of that subset's unshipped 9L.
- 19 of the 37 top ids were not in the file in August 2024. Those 19 are 27% of the main-cut unshipped 9L. Newness is part of the concentration. It is not the whole of it.
- The deletion flag and the active flag are not on any top id. Whatever they mean, they do not mark this loss.


### 22. Which months

**Question.** Is the loss stacked in a season, and is a bad month the same ids or a different set?

**How to read the output.** RTS ids are excluded. Fill rate is physical cases. The bars are unshipped 9L. The line is the fill rate. A month can carry a lot of 9L and still sit above 90% when ordered volume is high. The calendar block divides by the number of years, because August is in the file three times and every other month twice.


In [ ]:
ex = clean.loc[~clean["rts"]].copy()
monthly = ex.groupby(ex["month"].dt.to_period("M")).agg(
    ordered=("ordered", "sum"),
    unshipped=("unshipped", "sum"),
    uns9=("uns9", "sum"),
)
monthly["fill"] = 1 - monthly["unshipped"] / monthly["ordered"]
freq_ids = set(ranked.loc[ranked["quadrant"] == "FREQ_LARGE", "Source_Id"])
by_id_month = ex.groupby(["Source_Id", ex["month"].dt.to_period("M")])["uns9"].sum()
freq_month = by_id_month.loc[by_id_month.index.get_level_values(0).isin(freq_ids)].groupby(level=1).sum()
monthly["from_freq_large"] = freq_month.reindex(monthly.index).fillna(0) / monthly["uns9"]

shown = monthly.copy()
shown.index = shown.index.astype(str)
print(shown.round({"ordered": 0, "unshipped": 0, "uns9": 0, "fill": 3, "from_freq_large": 3}).to_string())
print(f"\nMonths under 90% fill: {int((monthly['fill'] < 0.9).sum())}")
top5 = monthly["uns9"].nlargest(5)
print(f"Five busiest months hold {pct(top5.sum() / monthly['uns9'].sum())} of RTS-excluded unshipped 9L")
print("Share of those months that sits on frequent-and-large ids:")
print(monthly.loc[top5.index, "from_freq_large"].round(3).to_string())

calendar = ex.copy()
calendar["cal"] = calendar["month"].dt.month
calendar["year"] = calendar["month"].dt.year
by_year = calendar.groupby(["cal", "year"]).agg(ordered=("ordered", "sum"), unshipped=("unshipped", "sum"), uns9=("uns9", "sum"))
rolled = by_year.groupby("cal").agg(years=("ordered", "size"), ordered=("ordered", "sum"), unshipped=("unshipped", "sum"), uns9=("uns9", "sum"))
rolled["fill"] = 1 - rolled["unshipped"] / rolled["ordered"]
rolled["avg_uns9"] = rolled["uns9"] / rolled["years"]
print("\nMonth of year, RTS excluded. avg_uns9 divides by the number of years.")
print(rolled.round({"ordered": 0, "unshipped": 0, "uns9": 0, "fill": 3, "avg_uns9": 0}).to_string())

org = ex.groupby(["Source_Id", "month", "Sales_Org"])["unshipped"].sum().unstack(fill_value=0)
org.columns = [f"org_{i}" for i in range(org.shape[1])]
if org.shape[1] != 2:
    raise RuntimeError("Expected two sales orgs")
both = (org["org_0"] > LOSS_FLOOR) & (org["org_1"] > LOSS_FLOOR)
id_month_uns = ex.groupby(["Source_Id", "month"])["unshipped"].sum()
print(
    "\nShare of unshipped cases in an id-month where both sales orgs miss:",
    pct(id_month_uns.loc[both].sum() / id_month_uns.sum()),
)

fig, axes = plt.subplots(2, 1, figsize=(9, 6.4))
axes[0].bar(range(len(monthly)), monthly["uns9"].to_numpy(), color="0.35")
axes[0].set_ylabel("Unshipped 9L")
axes[0].set_title(f"Distributor {DISTRIBUTOR.upper()}: unshipped 9L by month, RTS excluded")
axes[1].plot(range(len(monthly)), monthly["fill"].to_numpy(), marker="o", ms=3.5, color="0.15")
axes[1].axhline(0.90, color="0.45", lw=1)
axes[1].set_ylim(0.70, 1.01)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _pos: f"{value:.0%}"))
axes[1].set_ylabel("Fill rate")
for ax in axes:
    ax.set_xticks(range(len(monthly)))
    ax.set_xticklabels(monthly.index.astype(str), rotation=90)
fig.tight_layout()
plt.show()


**What it shows**

- Four months fill under 90%, RTS excluded: March 2025 (80%), July 2025 (86%), January 2026 (76%) and February 2026 (84%).
- The five months with the most unshipped 9L hold **51%** of the main-cut loss: February 2026, April 2025, March 2025, July 2025 and January 2026. April 2025 is second by lost 9L and still fills 92%, because the ordered volume is high. A low fill rate and a large loss are not the same list.
- In those five months, **79% to 92%** of the unshipped 9L sits on the frequent-and-large ids. The bad months are those ids having a worse month. They are not a separate set of one-off ids.
- Across the year, using the average so that August's third year does not win on a sum, the heavier months are March, February, April, July and January. May, October and November are light. This file does not show a May-June season.
- **74%** of the RTS-excluded unshipped cases fall in an id-month where both sales orgs miss the same id. That is still not evidence of one warehouse or of two. The question is open, and the clean rows stay split.


### 23. What this does not show

**Question.** What would be a wrong next sentence?

**How to read this.** No code. These are the limits of the cut above.

- The quadrant is not a root cause. Frequent and large is the buffer reading only. Rare and large is the supply or allocation reading only. Confirming either one needs forecasts, lead times and buffers. Those columns are not in this extract.
- The three RTS ids are flagged, not confirmed, and not removed. The totals with them are on the Analysis sheet.
- The 20 ids with no 9L factor, and the one id that points at two client SAP SKUs, are not given a guessed code.
- Distributor A is not cleaned. Its blank unshipped cells are still with the client.
- The single-brand pilot is a demo on 2026-10-30. It is not in this notebook, and the brand is not named here.


## Part 5 — The template rebuild

The October cut ranked a distributor item in one market on unshipped cases. The current file follows the frozen template: location is the sales org, the OOS measure is cases OOS, and December and January count half a month.

### 24. The cut


In [ ]:
from pathlib import Path
import pandas as pd

path = Path("data/cleaned") / "distributor_b_v1.xlsx"
clean = pd.read_excel(path, sheet_name="Clean")
top = pd.read_excel(path, sheet_name="Top_Cells")
qa = pd.read_excel(path, sheet_name="QA", header=None)
print("clean rows", len(clean), "columns", clean.shape[1])
print(qa.iloc[0:6, 0:3].to_string(index=False, header=False))
weighted = top["Weighted OOS 9L"]
print("loss cells", len(top))
print("top 80 cells", int((top["Top 80%"] == "Yes").sum()))
print("top 80 share", round(float(top.loc[top["Top 80%"] == "Yes", "Weighted OOS 9L"].sum() / weighted.sum()), 3))
print("oos rate", round(float(top["OOS 9L"].sum() / top["Ordered 9L"].sum()), 3))
print(top.groupby("Pattern")["Weighted OOS 9L"].sum().div(weighted.sum()).round(3).to_string())
print("locations", clean["Market"].nunique())
print("lag1 all blank", bool(clean["Fcst_Lag1"].isna().all()))


**What it shows.** 7,562 clean rows, the same count as the raw extract, and the three control differences are zero. 415 location × client SKU cells have OOS 9L. 53 of them carry 80.3% of the weighted OOS 9L. The OOS rate on this cut is 4.4%. Repeat cells carry 56.7% of the weighted loss. Both sales orgs are in the file. Lag-1 forecast is blank on every row.

### 25. What this file still does not show


- Bias is not calculated. The local snapshot extract has no rows for this distributor, and a missing extract is not treated as a zero forecast.
- The action lines are the template readings. They are not confirmed root causes. Current buffer and recommended buffer are empty.
- The 14 duplicate keys are kept, not summed. Whether each pair is two products or a rename is still open (`docs/open_questions.md`, question 16).
- The 20 item codes missing from Master Data, and the one code that points at two client SKUs, are still unresolved. They stay out of the Pareto.
- Distributor A is not cleaned.
